# Inspect Phoenix Local Dotplot Workflow

Use this notebook locally against the Phoenix mount at `/home/nnataren/mnt/phoenix_hpc/Banksy_py`.

It is set up to inspect the main files involved in scripts 02, 03, and 04, with a particular focus on checking whether the number of clusters in the clean expression object matches the source spatial object and the exported dotplot summaries.

In [ ]:
from pathlib import Path
import json

import anndata as ad
import pandas as pd

PHOENIX_ROOT = Path('/home/nnataren/mnt/phoenix_hpc/Banksy_py')
PHOENIX_ROOT

In [ ]:
sample = 'MF_skin_non_res'
resolution_label = '1.10'
resolution_file_label = '1p1'
cluster_column = 'labels_scaled_gaussian_pc20_nc0.20_r1.10'

paths = {
    'script_02': PHOENIX_ROOT / '02_create_expression_adata_with_banksy_clusters.py',
    'script_03': PHOENIX_ROOT / '03_export_dotplot_data_from_config.py',
    'script_04_local': PHOENIX_ROOT / '04_plot_multi_sample_dotplot_from_config_local.py',
    'readme_00_04': PHOENIX_ROOT / 'README_00_04_XENIUM_ANALYSIS_WORKFLOW.md',
    'config_02': PHOENIX_ROOT / 'config/02_create_expression/vbct_small/MF_skin_non_res.json',
    'config_03_all_genes': PHOENIX_ROOT / 'config/03_export_summary/archive/vbct_small_legacy_script02/all_genes/MF_skin_non_res_all_genes.json',
    'config_03_canonical': PHOENIX_ROOT / 'config/03_export_summary/archive/vbct_small_legacy_script02/canonical_markers/MF_skin_non_res_canonical_markers.json',
    'config_04_local_all_genes': PHOENIX_ROOT / 'config/04_plot_dotplot/local/all_genes_multi_sample_local.json',
    'config_04_local_canonical': PHOENIX_ROOT / 'config/04_plot_dotplot/local/canonical_markers_multi_sample_local.json',
    'clean_expression_h5ad': PHOENIX_ROOT / f'data/xenium/processed/{sample}/{sample}_normalised_log1p_with_banksy_clusters.h5ad',
    'spatial_h5ad': PHOENIX_ROOT / f'data/xenium/processed/{sample}/adata_spatial_{sample}_{resolution_file_label}.h5ad',
    'all_genes_export_csv': PHOENIX_ROOT / f'data/xenium/processed/cross_sample_dotplot_exports/all_genes_all_res/{sample}_all_genes_dotplot_summary.csv',
    'canonical_export_csv': PHOENIX_ROOT / f'data/xenium/processed/cross_sample_dotplot_exports/canonical_all_res/{sample}_canonical_markers_dotplot_summary.csv',
}

pd.DataFrame(
    {
        'name': list(paths.keys()),
        'path': [str(path) for path in paths.values()],
        'exists': [path.exists() for path in paths.values()],
    }
)

In [ ]:
for key in ['config_02', 'config_03_all_genes', 'config_03_canonical', 'config_04_local_all_genes']:
    print(f'\n--- {key} ---')
    with open(paths[key]) as f:
        cfg = json.load(f)
    print(json.dumps(cfg, indent=2)[:4000])

In [ ]:
expr = ad.read_h5ad(paths['clean_expression_h5ad'])
expr

In [ ]:
expr.obs.columns.tolist()

In [ ]:
expr_cluster_counts = expr.obs[cluster_column].astype(str).value_counts().sort_index()
print('Unique clusters in clean expression object:', expr_cluster_counts.shape[0])
expr_cluster_counts

In [ ]:
spatial = ad.read_h5ad(paths['spatial_h5ad'])
spatial

In [ ]:
spatial_cluster_counts = spatial.obs[cluster_column].astype(str).value_counts().sort_index()
print('Unique clusters in source spatial object:', spatial_cluster_counts.shape[0])
spatial_cluster_counts

In [ ]:
cluster_compare = pd.concat(
    [
        expr_cluster_counts.rename('clean_expression_n_cells'),
        spatial_cluster_counts.rename('spatial_n_cells'),
    ],
    axis=1,
).fillna(0).astype(int)

cluster_compare['present_in_clean_expression'] = cluster_compare['clean_expression_n_cells'] > 0
cluster_compare['present_in_spatial'] = cluster_compare['spatial_n_cells'] > 0
cluster_compare

In [ ]:
all_genes_df = pd.read_csv(paths['all_genes_export_csv'])
all_genes_df.head()

In [ ]:
all_genes_subset = all_genes_df[all_genes_df['resolution'].astype(str) == resolution_label].copy()
export_cluster_counts = (
    all_genes_subset[['cluster_id', 'sample_cluster']]
    .drop_duplicates()
    .sort_values('cluster_id')
)

print('Unique clusters in script 03 all-genes export at this resolution:', export_cluster_counts.shape[0])
export_cluster_counts

In [ ]:
canonical_df = pd.read_csv(paths['canonical_export_csv'])
canonical_subset = canonical_df[canonical_df['resolution'].astype(str) == resolution_label].copy()
print('Unique clusters in script 03 canonical export at this resolution:', canonical_subset['cluster_id'].astype(str).nunique())

In [ ]:
summary = {
    'clean_expression_unique_clusters': int(expr_cluster_counts.shape[0]),
    'spatial_unique_clusters': int(spatial_cluster_counts.shape[0]),
    'all_genes_export_unique_clusters': int(export_cluster_counts.shape[0]),
    'canonical_export_unique_clusters': int(canonical_subset['cluster_id'].astype(str).nunique()),
}
summary

## How to use this notebook

1. Run the first few cells to confirm the Phoenix-mounted files exist locally.
2. Check whether the clean expression h5ad and the source spatial h5ad have the same number of unique clusters for the chosen `cluster_column`.
3. Compare those counts against the script 03 exports.
4. If the spatial object has more clusters than the clean expression object, the mismatch was introduced during script 02 transfer into the clean expression object.
5. If the clean expression object and script 03 export differ, the mismatch was introduced during script 03 export.